# Course 1

## Expected Value Calculator


The **expected value** (or mean) of a discrete random variable is the weighted average of all possible values, where each value is weighted by its probability:

$$E[X] = \sum_{i} x_i \cdot \Pr[X = x_i]$$

The function below takes two lists:
- `amounts` — the possible values (e.g. dollar payouts)
- `probabilities` — the probability of each value occurring

It multiplies each amount by its probability and sums the results.

### Example — Horse Race Payout

| Horse | A | B | C | D | E | F |
|-------|------|------|--------|------|------|------|
| Payout | $9,000 | $4,500 | $18,000 | $6,000 | $3,000 | $4,500 |
| Probability | 0.1 | 0.2 | 0.05 | 0.15 | 0.3 | 0.2 |

$$E[P] = 0.1 \times 9000 + 0.2 \times 4500 + 0.05 \times 18000 + 0.15 \times 6000 + 0.3 \times 3000 + 0.2 \times 4500 = 5400$$

The house collected $6,000 (6 × $1,000) but pays out $5,400 on average — a $600 expected

### Expected Value formula

In [38]:
def expected_value(amounts, probabilities):
    if len(amounts) != len(probabilities):
        raise ValueError("amounts and probabilities must have the same length")
    return sum(a * p for a, p in zip(amounts, probabilities))

### Horse race example

In [39]:
payouts       = [9000, 4500, 18000, 6000, 3000, 4500]
probabilities = [0.1,  0.2,  0.05,  0.15, 0.3,  0.2]
bet_per_horse = 1000

ev = expected_value(payouts, probabilities)
house_collected = len(payouts) * bet_per_horse
print(f"Expected payout: ${ev:,.0f}")
print(f"House collected: ${house_collected:,.0f}")
print(f"House expected profit: ${house_collected - ev:,.0f}")

Expected payout: $5,400
House collected: $6,000
House expected profit: $600


## Expected Value - Adding Odds

Given estimated probabilities $\tilde{p}_i$ for each horse and a desired **house edge**, we set the payout odds so the house keeps a fixed percentage of the total bets.

The payout factor is $X = 1 - \text{house\_edge}$. For a 10% edge, $X = 0.9$.

The trivial solution sets each payout multiplier as:

$$o_i = \frac{X}{\tilde{p}_i}$$

This guarantees:

$$E[\tilde{O}] = \sum_i o_i \cdot \tilde{p}_i = \sum_i X = n \cdot X$$

So if $n$ horses each get a 1\$ bet, the house collects $n$\$ and pays out $n \cdot X$\$ on average.

### Rigging Odds function

In [40]:
def rig_odds(probabilities, house_edge=0.10, bet_per_horse=1):
    """Compute payout multipliers that guarantee a desired house edge.
    
    o_i = X / p_i  where X = 1 - house_edge
    """
    x = 1 - house_edge
    payout_multipliers = [x / p for p in probabilities]
    payouts = [o * bet_per_horse for o in payout_multipliers]
    return payouts

### Example: 6 horses, 10% house edge, $1000 bet per horse

In [41]:
probabilities = [0.1, 0.2, 0.05, 0.15, 0.3, 0.2]
house_edge = 0.10
bet_per_horse = 1000

payouts = rig_odds(probabilities, house_edge, bet_per_horse)
n = len(probabilities)
house_collected = n * bet_per_horse
ev = expected_value(payouts, probabilities)

print(f"House edge: {house_edge:.0%}")
print(f"Payout factor X: {1 - house_edge}")
print(f"Bet per horse: ${bet_per_horse:,.0f}")
print()
for i, (p, payout) in enumerate(zip(probabilities, payouts)):
    print(f"  Horse {i+1}: p={p:.2f}  →  payout = ${payout:,.0f}  (multiplier = {payout/bet_per_horse:.1f}x)")
print()
print(f"House collected: ${house_collected:,.0f}")
print(f"Expected payout: ${ev:,.0f}")
print(f"House expected profit: ${house_collected - ev:,.0f} ({(house_collected - ev)/house_collected:.0%})")

House edge: 10%
Payout factor X: 0.9
Bet per horse: $1,000

  Horse 1: p=0.10  →  payout = $9,000  (multiplier = 9.0x)
  Horse 2: p=0.20  →  payout = $4,500  (multiplier = 4.5x)
  Horse 3: p=0.05  →  payout = $18,000  (multiplier = 18.0x)
  Horse 4: p=0.15  →  payout = $6,000  (multiplier = 6.0x)
  Horse 5: p=0.30  →  payout = $3,000  (multiplier = 3.0x)
  Horse 6: p=0.20  →  payout = $4,500  (multiplier = 4.5x)

House collected: $6,000
Expected payout: $5,400
House expected profit: $600 (10%)


### Unbalanced Book & Odds Quality

When bets are **not** balanced (different $b_i$ on each horse), the expected payout becomes:

$$E[P] = \sum_i (1-f) \frac{b_i}{\tilde{p}_i} \cdot p_i = (1-f) \cdot \sum_i b_i \cdot \frac{p_i}{\tilde{p}_i}$$

This decomposes into:
- **$(1-f)$** — the payout fraction (house keeps $f$)
- **$\sum b_i$** — the predicted **take-in** (total bets)
- **$\frac{p_i}{\tilde{p}_i}$** — the **odds quality** per horse (how wrong our probability estimates are)

If $\tilde{p}_i = p_i$ (perfect estimates), the ratio is 1 and the house edge is exactly $f$.  
If our estimates are off, the worst case is bounded by $\max_i \frac{p_i}{\tilde{p}_i}$, and the house can lose money.

**That's why odds change** as bets come in — the house adjusts $o_i$ by weighing in the actual bet amounts $b_i$.

In [42]:
def rig_odds_unbalanced(p_estimated, p_real, bets, house_edge=0.10):
    """Compute expected payout with unbalanced book and imperfect estimates.
    
    o_i = (1 - f) / p_estimated_i   (odds set using estimates)
    Actual E[P] = sum( o_i * b_i * p_real_i )  (reality uses true probabilities)
    """
    f = house_edge
    x = 1 - f
    
    # Odds set by the house using estimated probabilities
    odds = [x / p_est for p_est in p_estimated]
    
    # Actual expected payout (weighted by real probabilities)
    ev_payout = sum(o * b * p for o, b, p in zip(odds, bets, p_real))
    
    total_bets = sum(bets)
    odds_quality = [p / p_est for p, p_est in zip(p_real, p_estimated)]
    
    return odds, ev_payout, total_bets, odds_quality

# Example: house estimates differ from reality, unbalanced bets
p_estimated = [0.10, 0.20, 0.05, 0.15, 0.30, 0.20]  # house estimates
p_real      = [0.08, 0.18, 0.10, 0.14, 0.25, 0.25]  # true probabilities
bets        = [500,  2000, 300,  1000, 1500, 700 ]   # actual bets per horse
house_edge  = 0.10

odds, ev_payout, total_bets, quality = rig_odds_unbalanced(
    p_estimated, p_real, bets, house_edge
)

print(f"House edge (target): {house_edge:.0%}")
print(f"Total bets collected: ${total_bets:,.0f}")
print()
print(f"{'Horse':<8} {'p_est':>6} {'p_real':>6} {'bet':>8} {'odds':>6} {'quality':>8}")
print("-" * 50)
for i in range(len(bets)):
    print(f"  {i+1:<6} {p_estimated[i]:>6.2f} {p_real[i]:>6.2f} ${bets[i]:>6,} {odds[i]:>6.1f}x {quality[i]:>7.2f}")
print()
print(f"Expected payout (real): ${ev_payout:,.0f}")
print(f"Expected profit: ${total_bets - ev_payout:,.0f} ({(total_bets - ev_payout)/total_bets:.1%})")
print(f"Worst odds quality: {max(quality):.2f} (horse {quality.index(max(quality))+1})")

if max(quality) > 1:
    print(f"\n⚠ House overestimated some horses — actual edge is worse than {house_edge:.0%}")

House edge (target): 10%
Total bets collected: $6,000

Horse     p_est p_real      bet   odds  quality
--------------------------------------------------
  1        0.10   0.08 $   500    9.0x    0.80
  2        0.20   0.18 $ 2,000    4.5x    0.90
  3        0.05   0.10 $   300   18.0x    2.00
  4        0.15   0.14 $ 1,000    6.0x    0.93
  5        0.30   0.25 $ 1,500    3.0x    0.83
  6        0.20   0.25 $   700    4.5x    1.25

Expected payout (real): $5,272
Expected profit: $728 (12.1%)
Worst odds quality: 2.00 (horse 3)

⚠ House overestimated some horses — actual edge is worse than 10%


## Dynamic Odds with Existing Liabilities

In reality, the house doesn't start from zero. It already has:
- **$L_i$** — liability on horse $i$ (what it must pay out from previous bets)
- **$B_i$** — total bets already received on horse $i$

Now it must set new odds $o_i$ for the estimated future bets $\tilde{b}_i$.

If horse $i$ wins, the house pays out:

$$L_i + \tilde{b}_i \cdot o_i$$

So the payout random variable is:

$$\tilde{P} \sim \frac{L_1 + \tilde{b}_1 o_1}{\tilde{p}_1} \quad \frac{L_2 + \tilde{b}_2 o_2}{\tilde{p}_2} \quad \dots \quad \frac{L_n + \tilde{b}_n o_n}{\tilde{p}_n}$$

And we want: $E[\tilde{P}] = (1-f) \sum_i (B_i + \tilde{b}_i)$

#### Two solutions:

**Solution 1 (risky):** Each horse pays out proportionally to the bets on it:
$$L_i + \tilde{b}_i o_i = \frac{(1-f)}{\tilde{p}_i}(B_i + \tilde{b}_i)$$
Problem: if the fans are right and the heavily-bet horse wins, the house loses big.

**Solution 2 (safe):** Each horse pays out the **same flat amount**:
$$L_i + \tilde{b}_i o_i = \frac{(1-f) \sum_j (B_j + \tilde{b}_j)}{n}$$
The house wins the same **regardless of which horse wins** — zero risk.

In [43]:
def rig_odds_with_liabilities(liabilities, past_bets, future_bets, p_estimated, house_edge=0.10, strategy="safe"):
    """Compute new odds given existing liabilities and expected future bets.
    
    Args:
        liabilities: L_i - current liability on each horse (from past bets at old odds)
        past_bets:   B_i - total past bets received on each horse
        future_bets: b_i - estimated future bets on each horse
        p_estimated: p_i - estimated win probability for each horse
        house_edge:  f   - desired house edge fraction
        strategy:    "safe" (flat payout) or "risky" (proportional to bets)
    
    Returns: new odds o_i for future bets
    """
    f = house_edge
    n = len(liabilities)
    total_take_in = sum(B + b for B, b in zip(past_bets, future_bets))
    target_payout = (1 - f) * total_take_in
    
    if strategy == "safe":
        # Solution 2: flat payout — every horse pays the same
        # L_i + b_i * o_i = target_payout / n
        flat = target_payout / n
        odds = [(flat - L) / b if b > 0 else 0
                for L, b in zip(liabilities, future_bets)]
    else:
        # Solution 1: proportional — payout weighted by bets on that horse
        # L_i + b_i * o_i = (1-f) / p_i * (B_i + b_i)
        odds = [((1 - f) / p * (B + b) - L) / b if b > 0 else 0
                for L, B, b, p in zip(liabilities, past_bets, future_bets, p_estimated)]
    
    # What the house actually pays if horse i wins
    actual_payouts = [L + b * o for L, b, o in zip(liabilities, future_bets, odds)]
    
    return odds, actual_payouts, target_payout

# Example
n = 6
p_estimated = [0.10, 0.20, 0.05, 0.15, 0.30, 0.20]
past_bets   = [800,  1500, 200,  600,  2000, 900 ]  # B_i
liabilities = [7200, 6750, 3400, 3600, 6000, 4050]  # L_i (from old odds)
future_bets = [200,  500,  100,  400,  500,  300 ]  # expected new bets
house_edge  = 0.10

for strategy in ["safe", "risky"]:
    odds, payouts, target = rig_odds_with_liabilities(
        liabilities, past_bets, future_bets, p_estimated, house_edge, strategy
    )
    total_in = sum(B + b for B, b in zip(past_bets, future_bets))
    
    print(f"=== Strategy: {strategy.upper()} ===")
    print(f"Total take-in: ${total_in:,.0f}  |  Target payout: ${target:,.0f}")
    print()
    print(f"{'Horse':<7} {'L_i':>7} {'B_i':>7} {'b_i':>7} {'new o_i':>8} {'payout if wins':>15}")
    print("-" * 55)
    for i in range(n):
        print(f"  {i+1:<5} ${liabilities[i]:>6,} ${past_bets[i]:>6,} ${future_bets[i]:>6,} {odds[i]:>7.1f}x ${payouts[i]:>13,.0f}")
    
    payout_spread = max(payouts) - min(payouts)
    print(f"\nPayout spread: ${payout_spread:,.0f} (0 = perfectly hedged)")
    print(f"House profit if ANY horse wins: ${total_in - min(payouts):,.0f} to ${total_in - max(payouts):,.0f}")
    print()

=== Strategy: SAFE ===
Total take-in: $8,000  |  Target payout: $7,200

Horse       L_i     B_i     b_i  new o_i  payout if wins
-------------------------------------------------------
  1     $ 7,200 $   800 $   200   -30.0x $        1,200
  2     $ 6,750 $ 1,500 $   500   -11.1x $        1,200
  3     $ 3,400 $   200 $   100   -22.0x $        1,200
  4     $ 3,600 $   600 $   400    -6.0x $        1,200
  5     $ 6,000 $ 2,000 $   500    -9.6x $        1,200
  6     $ 4,050 $   900 $   300    -9.5x $        1,200

Payout spread: $0 (0 = perfectly hedged)
House profit if ANY horse wins: $6,800 to $6,800

=== Strategy: RISKY ===
Total take-in: $8,000  |  Target payout: $7,200

Horse       L_i     B_i     b_i  new o_i  payout if wins
-------------------------------------------------------
  1     $ 7,200 $   800 $   200     9.0x $        9,000
  2     $ 6,750 $ 1,500 $   500     4.5x $        9,000
  3     $ 3,400 $   200 $   100    20.0x $        5,400
  4     $ 3,600 $   600 $   400  